# Level 2 — Task 3: Statistical Analysis for Business Decisions
**Internship:** Codveda Technology — Business Analytics  
**Objective:** Apply statistical methods to support business decision-making — hypothesis testing, probability distributions, A/B testing, and confidence intervals.  
**Datasets Used:**
- `churn_cleaned.csv` — Telecom customer churn
- `Sentiment_dataset.csv` — Social media sentiment data

---

## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import (ttest_ind, ttest_rel, chi2_contingency,
                          norm, binom, poisson, expon)
import warnings, os
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
np.random.seed(42)

print('Libraries loaded ✓')

---
## 1. Load Data

In [ ]:
DATA_DIR = '../data'

churn     = pd.read_csv(os.path.join(DATA_DIR, 'churn_cleaned.csv'))
sentiment = pd.read_csv(os.path.join(DATA_DIR, '3__Sentiment_dataset.csv'))

# Prep churn labels
churn['Churn_label'] = churn['Churn'].map({1:'Churned', 0:'Retained',
                                            True:'Churned', False:'Retained'})
churned  = churn[churn['Churn_label'] == 'Churned']
retained = churn[churn['Churn_label'] == 'Retained']

# Drop index columns from sentiment
sentiment = sentiment.drop(columns=[c for c in sentiment.columns
                                    if 'Unnamed' in c], errors='ignore')

print(f'Churn    : {churn.shape}  | Churned: {len(churned):,}  Retained: {len(retained):,}')
print(f'Sentiment: {sentiment.shape}')
display(sentiment.head(3))

---
## 2. Probability Distributions
Understanding which distribution best fits the data informs risk models and forecasting.
We test three common business distributions: **Normal**, **Poisson**, and **Exponential**.

In [ ]:
def fit_and_plot_distribution(data, col_name, dist_name='normal', color='#3498db', filename=None):
    """
    Reusable: fits a theoretical distribution to data and plots the overlay.
    dist_name: 'normal', 'poisson', or 'exponential'
    """
    fig, ax = plt.subplots(figsize=(9, 5))
    values = data.dropna().values

    ax.hist(values, bins=40, density=True, color=color,
            edgecolor='white', alpha=0.7, label='Observed data')

    x = np.linspace(values.min(), values.max(), 300)

    if dist_name == 'normal':
        mu, sigma = norm.fit(values)
        ax.plot(x, norm.pdf(x, mu, sigma), 'r-', linewidth=2.5,
                label=f'Normal fit\nμ={mu:.2f}, σ={sigma:.2f}')
        # KS test
        ks_stat, ks_p = stats.kstest(values, 'norm', args=(mu, sigma))
        result = f'KS stat={ks_stat:.4f}, p={ks_p:.4f}'

    elif dist_name == 'poisson':
        lam = values.mean()
        x_int = np.arange(int(values.min()), int(values.max())+1)
        ax.vlines(x_int, 0, poisson.pmf(x_int, lam), color='red',
                  linewidth=2, label=f'Poisson fit\nλ={lam:.2f}')
        result = f'λ (mean) = {lam:.3f}'

    elif dist_name == 'exponential':
        loc, scale = expon.fit(values, floc=0)
        ax.plot(x, expon.pdf(x, loc, scale), 'r-', linewidth=2.5,
                label=f'Exponential fit\nλ={1/scale:.4f}')
        result = f'Rate λ = {1/scale:.4f}'

    ax.set_title(f'{dist_name.title()} Distribution Fit — {col_name}\n{result}',
                 fontweight='bold')
    ax.set_xlabel(col_name)
    ax.set_ylabel('Density / Probability')
    ax.legend()
    plt.tight_layout()
    if filename:
        plt.savefig(filename, bbox_inches='tight')
    plt.show()

# Day minutes — normal distribution
fit_and_plot_distribution(
    churn['Total day minutes'], 'Total Day Minutes',
    dist_name='normal', color='#3498db',
    filename='dist_normal_day_minutes.png')

# Service calls — Poisson (count data)
fit_and_plot_distribution(
    churn['Customer service calls'], 'Customer Service Calls',
    dist_name='poisson', color='#9b59b6',
    filename='dist_poisson_service_calls.png')

# Intl calls — exponential
fit_and_plot_distribution(
    churn['Total intl minutes'], 'Total International Minutes',
    dist_name='exponential', color='#e67e22',
    filename='dist_exponential_intl.png')

---
## 3. Hypothesis Testing — Independent Samples t-Test

**Business Question:** Do churned customers use significantly more day minutes than retained customers?

- **H₀ (Null):** μ_churned = μ_retained (no difference in day usage)
- **H₁ (Alternative):** μ_churned ≠ μ_retained (significant difference)
- **Significance level α = 0.05**

In [ ]:
def run_ttest(group_a, group_b, label_a, label_b, metric, alpha=0.05):
    """
    Reusable: performs independent-samples t-test with full interpretation.
    Returns t-statistic, p-value, and conclusion.
    """
    a = group_a[metric].dropna()
    b = group_b[metric].dropna()

    t_stat, p_value = ttest_ind(a, b, equal_var=False)  # Welch's t-test
    cohen_d = (a.mean() - b.mean()) / np.sqrt((a.std()**2 + b.std()**2) / 2)

    print(f'{'─'*55}')
    print(f'  t-Test: {label_a} vs {label_b} — {metric}')
    print(f'{'─'*55}')
    print(f'  {label_a:12} mean : {a.mean():.3f}  (n={len(a):,})')
    print(f'  {label_b:12} mean : {b.mean():.3f}  (n={len(b):,})')
    print(f'  t-statistic      : {t_stat:.4f}')
    print(f'  p-value          : {p_value:.6f}')
    print(f"  Cohen's d        : {cohen_d:.4f}  ({'large' if abs(cohen_d)>0.8 else 'medium' if abs(cohen_d)>0.5 else 'small'} effect)")
    print()
    if p_value < alpha:
        print(f'  ✅ REJECT H₀ — Statistically significant difference (p < {alpha})')
        conclusion = 'significant'
    else:
        print(f'  ❌ FAIL TO REJECT H₀ — No significant difference (p ≥ {alpha})')
        conclusion = 'not significant'
    print()
    return t_stat, p_value, conclusion

metrics_to_test = [
    'Total day minutes',
    'Total day charge',
    'Total intl minutes',
    'Account length'
]
results = {}
for m in metrics_to_test:
    t, p, c = run_ttest(churned, retained, 'Churned', 'Retained', m)
    results[m] = {'t_stat': round(t, 4), 'p_value': round(p, 6), 'conclusion': c}

In [ ]:
# Visualise distributions side by side
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, metric in zip(axes, metrics_to_test):
    ax.hist(retained[metric].dropna(), bins=35, alpha=0.6,
            color='#3498db', edgecolor='white', density=True, label='Retained')
    ax.hist(churned[metric].dropna(),  bins=35, alpha=0.6,
            color='#e74c3c', edgecolor='white', density=True, label='Churned')
    ax.axvline(retained[metric].mean(), color='#2980b9', linestyle='--', linewidth=1.8)
    ax.axvline(churned[metric].mean(),  color='#c0392b', linestyle='--', linewidth=1.8)
    p_val = results[metric]['p_value']
    sig = '✅ Significant' if p_val < 0.05 else '❌ Not Significant'
    ax.set_title(f'{metric}\np={p_val:.4f}  {sig}', fontweight='bold', fontsize=9)
    ax.set_xlabel('Value'); ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('t-Test: Churned vs Retained — Feature Distributions',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ttest_distributions.png', bbox_inches='tight')
plt.show()

---
## 4. Chi-Square Test — Categorical Independence

**Business Question:** Is there a statistically significant association between having an International Plan and churning?

- **H₀:** Churn is independent of International Plan status
- **H₁:** Churn is associated with International Plan status

In [ ]:
def run_chi2_test(df, col1, col2, alpha=0.05):
    """
    Reusable: performs chi-square test of independence between two categorical columns.
    Returns chi2 statistic, p-value, and contingency table.
    """
    contingency = pd.crosstab(df[col1], df[col2])
    chi2, p, dof, expected = chi2_contingency(contingency)
    cramers_v = np.sqrt(chi2 / (len(df) * (min(contingency.shape) - 1)))

    print(f'{'─'*55}')
    print(f'  Chi-Square Test: {col1} vs {col2}')
    print(f'{'─'*55}')
    print(f'  Chi² statistic : {chi2:.4f}')
    print(f'  Degrees of freedom: {dof}')
    print(f'  p-value        : {p:.6f}')
    print(f"  Cramér's V     : {cramers_v:.4f}  ({'strong' if cramers_v>0.3 else 'moderate' if cramers_v>0.1 else 'weak'} association)")
    print()
    print('  Observed Contingency Table:')
    display(contingency)
    print()
    if p < alpha:
        print(f'  ✅ REJECT H₀ — Significant association exists (p < {alpha})')
    else:
        print(f'  ❌ FAIL TO REJECT H₀ — No significant association (p ≥ {alpha})')
    print()
    return chi2, p, contingency

# Encode for display
churn_display = churn.copy()
churn_display['International plan'] = churn_display['International plan'].map(
    {1:'Yes', 0:'No', 'Yes':'Yes', 'No':'No'})

chi2_intl, p_intl, ct_intl = run_chi2_test(
    churn_display, 'International plan', 'Churn_label')

chi2_vm, p_vm, ct_vm = run_chi2_test(
    churn_display, 'Voice mail plan', 'Churn_label')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (ct, title, p_val) in zip(axes, [
    (ct_intl, 'International Plan vs Churn', p_intl),
    (ct_vm,   'Voicemail Plan vs Churn',    p_vm)
]):
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct.plot(kind='bar', ax=ax, color=['#3498db','#e74c3c'],
                edgecolor='white', width=0.6)
    sig = '✅ p<0.05' if p_val < 0.05 else '❌ p≥0.05'
    ax.set_title(f'{title}\nChi-Square — {sig}  (p={p_val:.4f})', fontweight='bold', fontsize=10)
    ax.set_ylabel('% of Group')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=0)
    ax.legend(title='Status')

plt.suptitle('Chi-Square Test — Plan Type vs Churn Association',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('chi2_test.png', bbox_inches='tight')
plt.show()

---
## 5. A/B Testing — Marketing Campaign Simulation

**Business Scenario:** A telecom company runs a retention campaign. Group A receives a **10% discount offer** (control). Group B receives a **personalised call + 10% discount** (treatment). We test whether the treatment improves retention.

We simulate this using the real churn dataset: Group A = customers with ≤ 2 service calls, Group B = customers with 3+ service calls who received intervention.

In [ ]:
def run_ab_test(df, group_col, group_a_val, group_b_val,
                metric, alpha=0.05, label_a='Group A', label_b='Group B'):
    """
    Reusable: performs a full A/B test with t-test and effect size.
    Returns full results dictionary.
    """
    a = df[df[group_col] == group_a_val][metric].dropna()
    b = df[df[group_col] == group_b_val][metric].dropna()

    t_stat, p_val = ttest_ind(a, b, equal_var=False)
    lift = (b.mean() - a.mean()) / a.mean() * 100
    cohen_d = (b.mean() - a.mean()) / np.sqrt((a.std()**2 + b.std()**2) / 2)

    print(f'{'━'*55}')
    print(f'  A/B TEST RESULTS — {metric}')
    print(f'{'━'*55}')
    print(f'  {label_a:20}: n={len(a):,}  mean={a.mean():.3f}  std={a.std():.3f}')
    print(f'  {label_b:20}: n={len(b):,}  mean={b.mean():.3f}  std={b.std():.3f}')
    print(f'  Lift                : {lift:+.2f}%')
    print(f'  t-statistic         : {t_stat:.4f}')
    print(f'  p-value             : {p_val:.6f}')
    print(f"  Cohen's d           : {cohen_d:.4f}")
    print()
    if p_val < alpha:
        print(f'  ✅ STATISTICALLY SIGNIFICANT — Deploy treatment to all customers')
    else:
        print(f'  ❌ NOT SIGNIFICANT — Do not scale treatment yet')
    print()
    return {'mean_a': a.mean(), 'mean_b': b.mean(), 'lift': lift,
            't_stat': t_stat, 'p_value': p_val, 'cohen_d': cohen_d,
            'significant': p_val < alpha, 'n_a': len(a), 'n_b': len(b)}

# Simulate: Group A = customers with intl plan=0, Group B = intl plan=1
# Metric: day minutes (proxy for engagement after campaign)
ab_result = run_ab_test(
    churn,
    group_col='International plan',
    group_a_val=0, group_b_val=1,
    metric='Total day minutes',
    label_a='Control (No Intl Plan)',
    label_b='Treatment (Intl Plan)'
)

In [ ]:
# A/B test visualisation
group_a = churn[churn['International plan'] == 0]['Total day minutes'].dropna()
group_b = churn[churn['International plan'] == 1]['Total day minutes'].dropna()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Distribution overlay
axes[0].hist(group_a, bins=35, alpha=0.6, color='#3498db',
             density=True, edgecolor='white', label='Control (A)')
axes[0].hist(group_b, bins=35, alpha=0.6, color='#e74c3c',
             density=True, edgecolor='white', label='Treatment (B)')
axes[0].axvline(group_a.mean(), color='#2980b9', linestyle='--', linewidth=2)
axes[0].axvline(group_b.mean(), color='#c0392b', linestyle='--', linewidth=2)
axes[0].set_title('Distribution Overlap', fontweight='bold')
axes[0].set_xlabel('Total Day Minutes')
axes[0].legend()

# Means with error bars
means  = [group_a.mean(), group_b.mean()]
errors = [group_a.std() / np.sqrt(len(group_a)),
          group_b.std() / np.sqrt(len(group_b))]
axes[1].bar(['Control (A)', 'Treatment (B)'], means,
            yerr=errors, color=['#3498db','#e74c3c'],
            capsize=6, edgecolor='white', width=0.5)
for i, (m, e) in enumerate(zip(means, errors)):
    axes[1].text(i, m + e + 0.5, f'{m:.1f}', ha='center', fontweight='bold')
axes[1].set_title('Group Means ± SE', fontweight='bold')
axes[1].set_ylabel('Mean Day Minutes')

# Summary card
axes[2].axis('off')
sig_color = '#27ae60' if ab_result['significant'] else '#e74c3c'
sig_text  = 'SIGNIFICANT ✅' if ab_result['significant'] else 'NOT SIGNIFICANT ❌'
summary_text = (
    f"A/B TEST SUMMARY\n\n"
    f"Control mean  : {ab_result['mean_a']:.2f}\n"
    f"Treatment mean: {ab_result['mean_b']:.2f}\n"
    f"Lift          : {ab_result['lift']:+.2f}%\n"
    f"p-value       : {ab_result['p_value']:.4f}\n"
    f"Cohen's d     : {ab_result['cohen_d']:.4f}\n\n"
    f"Result: {sig_text}"
)
axes[2].text(0.5, 0.5, summary_text, transform=axes[2].transAxes,
             fontsize=11, verticalalignment='center', horizontalalignment='center',
             bbox=dict(boxstyle='round,pad=0.8', facecolor='#f8f9fa', edgecolor=sig_color, linewidth=2),
             fontfamily='monospace')

plt.suptitle('A/B Test — Retention Campaign Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ab_test_results.png', bbox_inches='tight')
plt.show()

---
## 6. Confidence Intervals & Margin of Error

A confidence interval tells us the range within which the true population parameter lies, with a given level of certainty. We compute 95% CIs for key business metrics.

In [ ]:
def confidence_interval(data, confidence=0.95):
    """
    Reusable: computes confidence interval for a sample mean.
    Returns (mean, lower_bound, upper_bound, margin_of_error).
    """
    n    = len(data)
    mean = np.mean(data)
    se   = stats.sem(data)  # standard error
    ci   = stats.t.interval(confidence, df=n-1, loc=mean, scale=se)
    moe  = ci[1] - mean
    return mean, ci[0], ci[1], moe

metrics = {
    'Total day minutes'    : churn['Total day minutes'].dropna(),
    'Total day charge'     : churn['Total day charge'].dropna(),
    'Total intl minutes'   : churn['Total intl minutes'].dropna(),
    'Account length'       : churn['Account length'].dropna(),
    'Customer service calls': churn['Customer service calls'].dropna()
}

ci_results = []
print('95% Confidence Intervals for Key Business Metrics')
print('─'*65)
print(f'  {"Metric":<28} {"Mean":>8} {"Lower":>10} {"Upper":>10} {"MoE":>8}')
print('─'*65)
for name, data in metrics.items():
    mean, lo, hi, moe = confidence_interval(data)
    print(f'  {name:<28} {mean:>8.2f} {lo:>10.2f} {hi:>10.2f} {moe:>8.4f}')
    ci_results.append({'metric': name, 'mean': mean, 'lower': lo,
                        'upper': hi, 'moe': moe})
print('─'*65)

In [ ]:
# CI forest plot
ci_df = pd.DataFrame(ci_results)
ci_df_norm = ci_df.copy()

fig, ax = plt.subplots(figsize=(10, 5))

for i, row in ci_df.iterrows():
    ax.plot([row['lower'], row['upper']], [i, i],
            color='#3498db', linewidth=3, solid_capstyle='round')
    ax.scatter(row['mean'], i, color='#e74c3c', s=80, zorder=5)
    ax.text(row['upper'] + 0.3, i,
            f"{row['mean']:.1f} ± {row['moe']:.2f}",
            va='center', fontsize=8.5)

ax.set_yticks(range(len(ci_df)))
ax.set_yticklabels(ci_df['metric'], fontsize=10)
ax.set_xlabel('Value', fontweight='bold')
ax.set_title('95% Confidence Intervals — Key Business Metrics',
             fontweight='bold', fontsize=13)
ax.legend(handles=[
    mpatches.Patch(color='#3498db', label='95% CI range'),
    plt.Line2D([0],[0], marker='o', color='w',
               markerfacecolor='#e74c3c', markersize=9, label='Sample mean')
], loc='lower right')
plt.tight_layout()
plt.savefig('confidence_intervals.png', bbox_inches='tight')
plt.show()

---
## 7. Sentiment Data — Platform & Engagement Analysis
Statistical analysis of social media engagement across platforms.

In [ ]:
# Clean sentiment data
sent = sentiment.copy()
sent['Retweets'] = pd.to_numeric(sent['Retweets'], errors='coerce').fillna(0)
sent['Likes']    = pd.to_numeric(sent['Likes'],    errors='coerce').fillna(0)

print('── Sentiment Distribution ──')
display(sent['Sentiment'].value_counts())

# Engagement by sentiment
print('\n── Avg Engagement by Sentiment ──')
display(sent.groupby('Sentiment')[['Retweets','Likes']].agg(['mean','std']).round(2))

In [ ]:
# T-test: Do positive posts get more likes than negative posts?
pos_likes = sent[sent['Sentiment']=='Positive']['Likes'].dropna()
neg_likes = sent[sent['Sentiment']=='Negative']['Likes'].dropna()

t, p = ttest_ind(pos_likes, neg_likes, equal_var=False)

print('── t-Test: Positive vs Negative Post Likes ──')
print(f'  Positive likes mean : {pos_likes.mean():.2f} (n={len(pos_likes)})')
print(f'  Negative likes mean : {neg_likes.mean():.2f} (n={len(neg_likes)})')
print(f'  t-statistic         : {t:.4f}')
print(f'  p-value             : {p:.4f}')
print(f'  Result: {"✅ Significant" if p<0.05 else "❌ Not significant"}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Sentiment distribution
sent_counts = sent['Sentiment'].value_counts()
colors_sent = ['#2ecc71','#e74c3c','#f39c12','#3498db']
axes[0].bar(sent_counts.index, sent_counts.values,
            color=colors_sent[:len(sent_counts)], edgecolor='white')
axes[0].set_title('Sentiment Distribution', fontweight='bold')
axes[0].set_ylabel('Post Count')

# Avg likes by sentiment
avg_likes = sent.groupby('Sentiment')['Likes'].mean()
axes[1].bar(avg_likes.index, avg_likes.values,
            color=colors_sent[:len(avg_likes)], edgecolor='white')
axes[1].set_title('Average Likes by Sentiment', fontweight='bold')
axes[1].set_ylabel('Avg Likes')

# Platform distribution
plat_counts = sent['Platform'].value_counts()
axes[2].pie(plat_counts.values, labels=plat_counts.index,
            autopct='%1.1f%%', startangle=90,
            colors=['#3498db','#e74c3c','#2ecc71','#f39c12','#9b59b6'][:len(plat_counts)])
axes[2].set_title('Posts by Platform', fontweight='bold')

plt.suptitle('Sentiment Dataset — Statistical Overview',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('sentiment_analysis.png', bbox_inches='tight')
plt.show()

---
## 8. Risk Analysis — Probability of Churn
Using the binomial distribution to estimate churn probability and business risk.

In [ ]:
# Churn probability from data
p_churn = churn['Churn'].mean()
n_customers = 1000  # hypothetical new customer batch

# Expected churners and variance
expected_churn = n_customers * p_churn
variance       = n_customers * p_churn * (1 - p_churn)
std_dev        = np.sqrt(variance)

# Probability of > 170 churners (risk threshold)
threshold = 170
p_exceed  = 1 - binom.cdf(threshold, n_customers, p_churn)

# Plot binomial distribution
x = np.arange(0, int(expected_churn * 2.5))
pmf = binom.pmf(x, n_customers, p_churn)

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x, pmf, color='#3498db', edgecolor='white', alpha=0.75, label='Binomial PMF')
ax.bar(x[x > threshold], pmf[x > threshold],
       color='#e74c3c', edgecolor='white', alpha=0.85,
       label=f'Risk zone (> {threshold} churners)')
ax.axvline(expected_churn, color='#2c3e50', linestyle='--',
           linewidth=2, label=f'Expected churners: {expected_churn:.0f}')
ax.set_title(f'Churn Risk Distribution — Binomial(n={n_customers}, p={p_churn:.3f})\n'
             f'P(churners > {threshold}) = {p_exceed:.4f}',
             fontweight='bold', fontsize=12)
ax.set_xlabel('Number of Churners (out of 1,000 customers)')
ax.set_ylabel('Probability')
ax.legend()
plt.tight_layout()
plt.savefig('churn_risk_distribution.png', bbox_inches='tight')
plt.show()

print(f'  Churn probability   : {p_churn:.4f} ({p_churn*100:.1f}%)')
print(f'  Expected churners   : {expected_churn:.1f} out of {n_customers}')
print(f'  Std deviation       : ±{std_dev:.1f} customers')
print(f'  P(churn > {threshold})    : {p_exceed:.4f} ({p_exceed*100:.2f}%)')

---
## 9. Statistical Findings Summary

| Test | Business Question | Result |
|------|-------------------|--------|
| **t-Test: Day Minutes** | Do churned customers use more minutes? | ✅ Yes — significant (p<0.05) |
| **t-Test: Account Length** | Do churned customers have shorter tenure? | See output above |
| **Chi-Square: Intl Plan** | Does intl plan predict churn? | ✅ Strong association |
| **Chi-Square: Voicemail** | Does voicemail predict churn? | See output above |
| **A/B Test** | Does treatment group differ on key metrics? | See output above |
| **Confidence Intervals** | What is the true population mean range? | 95% CI computed for all metrics |
| **Risk Distribution** | What is the churn risk for a 1,000-customer batch? | Binomial model computed |

> **Business Recommendation:** The statistical evidence strongly supports targeting customers with International Plans and high service call volumes for proactive retention. Day usage is a key predictor — high usage does not guarantee loyalty, and pricing should be reviewed for high-usage segments.